# evergreen_lab — 전략 비교 노트북

여러 전략을 같은 시장·비용으로 돌려 **어느 전략이 리스크 대비 잘 버는지** 비교한다. 현물(매수=풀, 매도=청산) 기준.

- 전략 추가: `evergreen_lab/strategies/`에 파일 1개 + `strategies/__init__.py`에 import 1줄.
- 실데이터는 `outputs/data/upbit-cache`에 CSV로 캐시된다(최초 실행만 Upbit API 호출). 먼저 `uv sync`.

## 1. 셋업 (프로젝트 경로 + import)

In [ ]:
import os, sys
from pathlib import Path

# 'evergreen_lab'를 담은 프로젝트 루트를 찾아 sys.path에 추가한다.
root = Path.cwd()
while root != root.parent and not (root / 'evergreen_lab').is_dir():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import evergreen_lab as lab
print('registered strategies:', lab.list_strategies())


## 2. v6 배포 모델 학습 + export (`v6.onnx`)

Java(`V6ModelConfig`)가 classpath에서 로드하는 **배포용 ONNX 모델**을 KRW-BTC 4시간봉 전체 이력으로
학습해 `src/main/resources/strategy-models/v6.onnx`에 쓴다(학습 진행 바 표시). 표준화+MLP가 그래프에
baking된 self-contained 파일이며, JSON 매니페스트는 쓰지 않는다(규칙 파라미터는 config로 관리). 이 셀만
돌리면 언제든 모델을 다시 학습·재생성한다. 출력된 `resolved rule_scale`을 config
`evergreen.trading.v6.rule-scale`에 반영해야 라이브 피처 분포가 학습과 일치한다.

In [ ]:
from datetime import datetime, timezone

# 배포용 v6 모델을 KRW-BTC 4시간봉 전체 이력으로 학습해 Java classpath 리소스에 쓴다.
# (Java V6ModelConfig가 이 onnx를 로드. JSON 매니페스트는 쓰지 않는다.) 비교(아래)와는 독립이다.
v6_bars = lab.load_candles(
    market='KRW-BTC',
    from_dt=datetime(2017, 1, 1, tzinfo=timezone.utc),   # 가능한 전체 이력
    to_dt=None,
    interval='minute_240',
)
onnx_out = root / 'src' / 'main' / 'resources' / 'strategy-models' / 'v6.onnx'
lab.export_v6(str(onnx_out), v6_bars, progress=True)
print(f'v6 학습 완료: {len(v6_bars)}봉 -> {onnx_out}')
print('※ 위에 출력된 resolved rule_scale 값을 config evergreen.trading.v6.rule-scale 에 반영하세요.')

## 3. 비교 설정

비교할 시장·기간·비용과 전략 목록을 여기서 정한다. 전략마다 권장 interval이 달라
(v6=`minute_240`, v1~v5=`days`) 바 개수·기간이 다르므로 **절대 수익의 직접 비교는 주의** —
리스크 조정 지표(calmar), 낙폭, 위험/수익 산점도로 함께 본다.

In [ ]:
from datetime import datetime, timezone

# 비교 공통 설정 — 여기만 바꾼다.
market = 'KRW-BTC'
from_dt = datetime(2021, 1, 1, tzinfo=timezone.utc)
to_dt = datetime(2023, 1, 1, tzinfo=timezone.utc)                          # None = 지금까지
cost = lab.Cost(fee_per_side=0.0005, slippage=0.0002)

# 비교할 전략과 각 전략의 권장 interval (v6=minute_240 룰, v1~v5=일봉 레짐 전략).
order = ['v6', 'v1', 'v2', 'v3', 'v4', 'v5']
recommended_interval = {'v6': 'minute_240', 'v1': 'days', 'v2': 'days', 'v3': 'days', 'v4': 'days', 'v5': 'days'}

## 4. 전략 비교 표

각 전략을 권장 interval로 돌려 핵심 지표를 한 표로 본다. `lab.evaluate_strategies(...)`가 **멀티코어
병렬**로 평가하고 **진행 바**를 보여준다(전략이 많을수록 병렬 이득이 크다). `calmar = CAGR / |MDD|`
(높을수록 리스크 대비 수익이 좋음) 기준으로 정렬한다. 이 셀이 만드는 `results`를 아래 대시보드가 그대로 쓴다.

In [ ]:
import pandas as pd

# 여러 전략을 멀티코어 병렬로 평가하고 진행 바를 표시한다.
# 첫 실행(캐시 없음)이면 parallel=False로 캐시를 먼저 채운 뒤 병렬로 재실행하는 게 안전하다.
results = lab.evaluate_strategies(
    {name: recommended_interval[name] for name in order},
    market=market, from_dt=from_dt, to_dt=to_dt, cost=cost,
    parallel=True, progress=True,
)

rows = []
for name in order:
    s = results[name].summary
    calmar = s.cagr / abs(s.mdd) if s.mdd else float('nan')
    rows.append({'strategy': name, 'interval': recommended_interval[name],
                 'total_return': s.total_return, 'buy_hold': s.buy_hold_return,
                 'cagr': s.cagr, 'mdd': s.mdd, 'calmar': calmar,
                 'round_trips': s.round_trips, 'win_rate': s.win_rate})
comparison_table = pd.DataFrame(rows).sort_values('calmar', ascending=False).reset_index(drop=True)
comparison_table

## 5. 비교 대시보드

`results`(위 표)로 세 가지를 한눈에 비교한다:

1. **자산 곡선** (로그) — 누가 더 벌었나 + Buy&Hold 참조.
2. **낙폭(drawdown)** — 누가 더 깊이·오래 빠졌나(리스크).
3. **위험/수익 산점도** — 가로축 최대낙폭 MDD, 세로축 CAGR. **좌상단(고수익·저위험)일수록 우수**,
   점 크기는 매매 횟수. 절대 수익 하나가 아니라 리스크까지 함께 보는 게 핵심.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 한글 폰트를 전역으로 설정한다. (rc_context는 with 블록이 끝난 뒤 IPython이 그림을 렌더할 때
# 되돌려져 폰트/마이너스 설정이 빠지므로, U+2212 글리프 경고와 한글 깨짐을 유발한다.)
installed = {f.name for f in font_manager.fontManager.ttflist}
korean_font = next((f for f in ['AppleGothic', 'Malgun Gothic', 'NanumGothic', 'Apple SD Gothic Neo'] if f in installed), None)
if korean_font:
    plt.rcParams['font.family'] = korean_font
plt.rcParams['axes.unicode_minus'] = False   # 마이너스에 ASCII '-' 사용(AppleGothic엔 U+2212 글리프가 없음)


def _drawdown_pct(bt_rows):
    """러닝 최고점 대비 하락률(%) 시리즈."""
    peak = float('-inf'); out = []
    for row in bt_rows:
        peak = max(peak, row.equity)
        out.append((row.equity / peak - 1.0) * 100 if peak > 0 else 0.0)
    return out


# constrained_layout: 수동 gridspec + sharex 조합에서도 tight_layout 경고 없이 여백을 맞춘다.
fig = plt.figure(figsize=(13, 12), constrained_layout=True)
gs = fig.add_gridspec(3, 1, height_ratios=[2.2, 1.6, 2.2])
ax_eq = fig.add_subplot(gs[0])
ax_dd = fig.add_subplot(gs[1], sharex=ax_eq)
ax_sc = fig.add_subplot(gs[2])
cmap = plt.get_cmap('tab10')

for i, name in enumerate(order):
    r = results[name]; s = r.summary
    ts = [row.timestamp for row in r.rows]
    calmar = s.cagr / abs(s.mdd) if s.mdd else float('nan')
    ax_eq.plot(ts, [row.equity for row in r.rows], lw=1.4, color=cmap(i),
               label=f"{name}  {s.total_return:+.0%}  (calmar {calmar:.2f})")
    ax_dd.plot(ts, _drawdown_pct(r.rows), lw=1.0, color=cmap(i), label=name)

# Buy & Hold 참조 (가장 긴 일봉 시리즈 기준)
bh = results.get('v1') or next(iter(results.values()))
ax_eq.plot([row.timestamp for row in bh.rows], [row.equity_bh for row in bh.rows],
           lw=1.2, color='k', alpha=0.5, ls='--', label=f"Buy & Hold  {bh.summary.buy_hold_return:+.0%}")

ax_eq.set_yscale('log')
ax_eq.set_title('① 자산 곡선 비교 (시작 = 1.0, 로그 스케일)')
ax_eq.set_ylabel('자산 (log)')
ax_eq.legend(ncol=2, fontsize=8)
ax_eq.grid(True, which='both', alpha=0.3)

ax_dd.set_title('② 전략별 낙폭(drawdown) 비교')
ax_dd.set_ylabel('낙폭 %')
ax_dd.legend(ncol=6, fontsize=8)
ax_dd.grid(True, alpha=0.3)

for i, name in enumerate(order):
    s = results[name].summary
    ax_sc.scatter(abs(s.mdd) * 100, s.cagr * 100, s=40 + s.round_trips * 3,
                  color=cmap(i), alpha=0.75, edgecolor='white', zorder=3)
    ax_sc.annotate(name, (abs(s.mdd) * 100, s.cagr * 100),
                   textcoords='offset points', xytext=(6, 4), fontsize=10)
ax_sc.axhline(0, color='k', lw=0.6, alpha=0.4)
ax_sc.set_title('③ 위험/수익 (좌상단=고수익·저위험이 우수, 점 크기=매매 횟수)')
ax_sc.set_xlabel('최대낙폭 MDD %  (오른쪽일수록 위험 큼)')
ax_sc.set_ylabel('연복리수익 CAGR %')
ax_sc.grid(True, alpha=0.3)